# PlastPrior: source evidence and hazard/exposure dimension assessment

This notebook presents source inspection, traceable record assembly, chemical identity resolution and aggregation by full InChIKey. Stage 3A screens structures and integrates the frozen VEGA reports with explicit identity-review flags. Stage 3B reproduces historical Pov/LRTP calculations and isolates unverified identities from analysis outputs. Stage 4A assembles basic source evidence with explicit field mapping and selection audits. Stage 4B adds traceable LitChemPlast associations and FCChumon human-detection evidence. Stage 4C adds traceable NORMAN predictions and GIC regulatory-list associations. Stages 5A and 5B evaluate persistence, mobility, bioaccumulation and toxicity as separate dimensions. Stage 6A audits exposure evidence eligibility. Stage 6B computes exposure dimensions with separate historical and qualified analysis results. Aggregate hazard and combined prioritisation are not implemented.

## Setup

Use the **PlastPrior (Python 3.12)** kernel, restart it, and run all cells from the project directory. The `README.md` describes installation, policies and validation. Inputs are read without modification; generated reports are written to `results/`.

Stage 4C retains NORMAN prediction provenance, GIC query paths and metadata limitations. These annotations do not establish chemical identity or experimental detection.


Stage 5A evaluates persistence and mobility in separate historical and isolated analysis views. Stage 5B adds bioaccumulation and toxicity under the same isolation policy. Stage 6B retains unavailable exposure components as missing and requires all five dimensions for a formal exposure total. No aggregate hazard or combined priority score is calculated.


In [ ]:
from pathlib import Path
from IPython.display import display

if not (Path.cwd() / "plastprior" / "inputs.py").is_file():
    raise RuntimeError("Run this notebook from the PlastPrior project directory.")

from plastprior.inputs import build_source_records, inspect_inputs
from plastprior.identity import resolve_identities
from plastprior.qsar import integrate_qsar
from plastprior.lrtp import integrate_lrtp
from plastprior.evidence import integrate_evidence
from plastprior.evidence_specialists import integrate_specialist_evidence
from plastprior.evidence_annotations import integrate_annotations
from plastprior.hazard import integrate_pm_hazard
from plastprior.hazard_bt import integrate_bt_hazard

DATA_ROOT = Path("data")
REPORT_DIR = Path("results")
from plastprior.exposure_inputs import integrate_exposure_evidence
import pandas as pd
from plastprior.exposure import integrate_exposure_scores


## Stage 1 — Input validation and ingestion

Inspect the declared source files and selected worksheets. Versions and snapshot dates are recorded metadata; **Not recorded** indicates a provenance gap. Parsed rows are source records, not unique chemicals, and must not be summed across overlapping inventories. Table roles distinguish inventory records, identity references, and literature links.

Stage 1 requires the source-file and table checks to pass. Auxiliary inputs are listed separately and receive **presence checks only**; their contents must be validated by the stages that use them.

In [ ]:
inspection = inspect_inputs(DATA_ROOT)
inspection.write_reports(REPORT_DIR)

display(inspection.sources[[
    "source", "filename", "version", "snapshot_date", "status"
]])
display(inspection.tables[[
    "source_id", "table_id", "table_role", "parsed_rows", "nonempty_rows", "columns", "status"
]])

**Review the inspection outcome.** Source errors stop this stage after the reports have been saved. Missing auxiliary inputs remain visible but do not invalidate source inspection. A successful inspection does not establish scientific validity or readiness for scoring.

Full provenance notes, references, observed SHA-256 fingerprints, sheet/header declarations, and diagnostics are retained in `results/source_inventory.csv`, `results/table_dimensions.csv`, and `results/supporting_inputs.csv`. These files are regenerated on each run. An observed fingerprint identifies file bytes; it does not independently certify a source version.

In [ ]:
display(inspection.supporting[["input_group", "status"]])
if not inspection.supporting_present:
    print("Supporting inputs are incomplete; see supporting_inputs.csv.")

for label, frame in [("Source files", inspection.sources), ("Source tables", inspection.tables)]:
    failures = frame.loc[frame["status"].ne("OK")]
    if not failures.empty:
        print(label)
        display(failures)

reader_notes = inspection.tables.loc[
    inspection.tables["reader_warnings"].ne(""), ["table_id", "reader_warnings"]
]
if not reader_notes.empty:
    print("Workbook-reader notes")
    display(reader_notes)

inspection.require_sources()
print("Stage 1 source inspection passed. Auxiliary content validation remains pending.")

## Stage 2 — Chemical identity resolution

**Source record assembly.**

Assemble the 11 inventory tables using the explicit mappings in `plastprior/sources.json`. PubChemLite, NORMAN SusDat and the LitChemPlast literature-link table are outside the candidate inventory. PlastChem uses isomeric SMILES when supplied, with canonical SMILES as the declared fallback. The historical MPLEACH rule requires both reported InChIKey and SMILES; excluded records remain in the exported table with reasons.

Availability counts refer to **included source records**, not unique chemicals or validated identifiers. Missingness is determined using the documented, case-sensitive token list after whitespace trimming. Original mapped values, including placeholders and unselected fallback values, are retained in `reported_fields_json`. An empty mapping means the field is not supplied by that source.

The full records, field mappings, summary and provenance manifest are saved under `results/stage2/`. Frozen-file fingerprints and record positions define record identifiers; native identifiers are retained where explicitly declared. This assembly step precedes chemical matching; its original records are retained throughout identity resolution.

In [ ]:
source_records = build_source_records(DATA_ROOT)
source_records.write_reports(REPORT_DIR / "stage2")

display(source_records.summary[[
    "source_id", "table_id", "parsed_records", "included_records", "excluded_records"
]])
display(source_records.summary[[
    "table_id", "included_records", "name_available", "cas_available",
    "smiles_available", "inchikey_available"
]])
excluded = source_records.records.loc[~source_records.records["included"]]
if excluded.empty:
    print("No inventory records were excluded under the declared inclusion rules.")
else:
    display(excluded.groupby(["table_id", "exclusion_reason"]).size().rename("records").reset_index())
print("Source record assembly complete; proceed to identity resolution below.")

**Resolve structures and retain all source links.** The versioned migration policy first uses acceptable reported SMILES, then full InChIKey lookup in PubChemLite, PlastChem and NORMAN, then CAS lookup only when eligible reference candidates agree on one full key. Reference mappings are explicit in `sources.json`. Tier A–D denotes route precedence, not a probability of correct identification.

Aggregation uses the full structure-derived InChIKey and preserves every member record. A parsed structure without a valid identity key is retained at record level but cannot enter the compound inventory. The non-isomeric QSAR key is an annotation only; it is not used to collapse stereoisomers. Name matching, desalting, tautomer normalisation, QSAR eligibility screening and scoring are not performed.

The summary separates matching routes from valid-key counts. Full record-level decisions, selected reference row identifiers, compound memberships, duplicate reference keys and CAS conflicts are exported under `results/stage2/`. See the README for the separate, fingerprinted historical regression command. A successful notebook execution alone does not establish agreement with the historical baseline.


In [ ]:
identities = resolve_identities(source_records, DATA_ROOT)
identities.write_reports(REPORT_DIR / "stage2")

for measure, count in identities.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(identities.summary)
display(identities.reference_summary)
print("Stage 2 complete. Review unresolved records and audit reports before proceeding to QSAR integration.")


## Stage 3A — Frozen QSAR evidence integration

Apply the historical structure-screening policy and retain the Stage 2 identity fields unchanged. Export one row per non-isomeric QSAR representation; full chemical identities may share a prediction without being merged. The frozen input batch and all 20 report fingerprints must match before row-based associations are imported.

Each model has explicit version, field, unit and quality mappings in `plastprior/qsar_models.json`. Missing ADI is not zero; low ADI is retained for later evidence assessment. Variable-width report rows retain their raw cells in the layout audit. No predictions are recalculated, and no Pov/LRTP or priority calculations are performed.

**Identity review is separate from historical reproduction.** Report structures are checked against the input keys. Known discrepancies retain their historical predictions but are marked `qsar_identity_verified=False` and `qsar_identity_status=review_required`. These values must not be used as identity-verified evidence. Formula/radical differences and parser diagnostics describe observed discrepancies; their processing origin and chemical correctness remain unconfirmed.

The table below displays every identity exception. The compact `identity_anomalies.csv`, complete `identity_anomaly_sources.csv` and `identity_anomaly_reports.csv` preserve the input structures, original source records, reference identifiers, report row locations and file fingerprints. All outputs are under `results/stage3/`. Use the README's Stage 3A regression command to compare with the frozen historical outputs; successful execution alone does not establish historical agreement or scientific correctness.


In [ ]:
qsar = integrate_qsar(identities, DATA_ROOT)
qsar.write_reports(REPORT_DIR / "stage3")

for measure, count in qsar.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(qsar.model_summary[[
    "model_name", "model_metadata", "rows", "identity_review_rows", "error_assessment_rows", "adi_provided"
]])
exceptions = qsar.identity_audit.loc[~qsar.identity_audit["identity_verified"]]
display(exceptions[[
    "vega_row_no", "input_formula", "report_formula", "input_radical_electrons",
    "report_radical_electrons", "reason", "identity_status"
]])
print("Stage 3A complete. Identity exceptions remain pending scientific review; Pov/LRTP has not been calculated.")


## Stage 3B — Pov/LRTP calculation and identity isolation

Use the six declared QSAR properties to reproduce the historical three-compartment calculation. Water and soil half-lives are converted from days to hours; logKaw is estimated from log10 vapour pressure in atm and negative-log10 solubility in mol/L. Environmental constants, reference lines and operation order follow the archived implementation.

**Historical reproduction and analysis availability are separate.** `lrtp_historical_results.csv` retains all results and emission-scenario evidence. `compound_lrtp.csv` withholds the five converted inputs, four numerical outputs and three threshold flags for unverified identities. Missing values are not zero or False. Historical calculation status, identity reasons and all source/report links remain traceable. No downstream scores are calculated.

Availability means that numerical and identity checks permit a result; it does not establish adequate QSAR reliability or independent agreement with the official OECD workbook. Low ADI and source warnings remain recorded for later evidence assessment. The table below displays all identity exclusions and their historical values explicitly for audit only.

Results are saved under `results/stage3/lrtp/`. The README provides the separate regression command, including upstream Stage 2/3A verification and recursive comparison of historical scenario results.


In [ ]:
lrtp = integrate_lrtp(REPORT_DIR / "stage3")
lrtp.write_reports(REPORT_DIR / "stage3" / "lrtp")

for measure, count in lrtp.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(lrtp.compounds.groupby(
    ["calculation_status", "qsar_identity_status", "lrtp_available"], dropna=False
).size().rename("compounds").reset_index())
excluded = lrtp.compounds.loc[
    lrtp.compounds["lrtp_exclusion_reason"].eq("identity_review_required"),
    ["inchikey", "vega_row_no", "qsar_identity_reason", "calculation_status", "lrtp_exclusion_reason"]
].merge(lrtp.historical[["inchikey", "LRTP_POV_days", "LRTP_CTD_km", "LRTP_TE_percent"]],
        on="inchikey", validate="one_to_one")
display(excluded.rename(columns={
    "LRTP_POV_days": "Historical Pov (days; audit only)",
    "LRTP_CTD_km": "Historical CTD (km; audit only)",
    "LRTP_TE_percent": "Historical TE (%; audit only)"
}))
print("Stage 3B complete. Identity-review results are withheld from analysis; historical values remain available for audit.")


## Stage 4A — Basic source evidence

Assemble the mapped hazard, exposure, regulatory and classification observations from the 11 inventory tables. `evidence_fields.json` declares source columns and precedence. The historical lookup tries InChIKey, CAS and SMILES in order, selects the first matching parsed row, and applies its identity gate. Every query, alternative match, field observation and exclusion remains traceable. SMILES text agreement does not establish chemical identity; database-derived identity fields do not replace Stage 2 identities.

The analysis table retains independent database observations for all compounds but withholds the 57 QSAR fields for the 15 review-required identities and preserves their 12 existing Pov/LRTP exclusions. Historical predictions remain in Stage 3 reports. Availability expresses an identity gate, not model reliability. Source-native hazard scores are imported observations; no new score is calculated.

Reports under `results/stage4/` include explicit field/index mappings, complete retained source rows, selection and exclusion audits, historical basic evidence, and the analysis table. Stage 4A uses the historical reader's type inference and missing-token handling; original source bytes and Stage 2 reported fields remain available. Specialist LitChemPlast/FCChumon resolution, analytical detectability and GIC integration are deferred.

Use the separate Stage 4A regression command in the README. It compares each basic evidence value and historical audit event against the frozen run, restoring the generic literature counts from its pre-enrichment audit. This is a comparison at the 4A boundary, not a claim that later Stage 4 enrichments have been reproduced.


In [ ]:
evidence = integrate_evidence(REPORT_DIR / "stage2", REPORT_DIR / "stage3" / "lrtp", DATA_ROOT)
evidence.write_reports(REPORT_DIR / "stage4")

for measure, count in evidence.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
import pandas as pd
attempts = pd.DataFrame(evidence.attempts)
display(attempts.groupby(["db", "identity_gate_status"]).size().unstack(fill_value=0))
display(evidence.field_mapping.groupby("source").agg(
    declared_field_mappings=("source_column", "size"),
    available_field_mappings=("column_present", "sum")
))
display(evidence.isolation.groupby(["exclusion_reason", "action"]).size().rename("field_entries").reset_index())
print("Stage 4A complete. Identity-review predictions remain isolated; specialist enrichment and scoring are deferred.")


## Stage 4B — Literature associations and human detection

Use every Stage 2 source member to reconstruct the compound CAS aliases and their mapping status. LitChemPlast uses exact full InChIKey matching before guarded CAS fallback for source rows without a valid full key. Counts distinguish unique `LitChem_id` relationships from unique `Lit_id` publications. Declared source counts are audited separately; they are not added to relationship counts.

FCChumon requires an unambiguous target CAS and exactly one sheet/CAS candidate. S1 database entries and S2 studies have different detection definitions. Counts use the historical within-sheet maximum; cross-sheet matches are rejected rather than summed. A selected source with no positive count is distinct from no match or a rejected association. The legacy zero/False fields remain in historical resolution records; the analysis detection flag is missing when evidence is unavailable.

The 15 QSAR identity exceptions retain their model exclusions. Independently matched literature or human-detection evidence remains reviewable using the canonical Stage 2 identity. Source membership is unchanged; specialist evidence availability is a separate annotation. No chemical identity is repaired or merged here.

Outputs under `results/stage4/4b/` retain all four source tables, CAS/member links, candidate selection and exclusion reasons, relationship-count contributions, numeric-conversion decisions and a dedicated identity-review subset. Stage 4A files remain unchanged. The separate README regression command compares every CAS annotation, specialist audit field and evidence JSON value against the frozen 4B boundary. Successful execution alone is not proof of historical agreement. Stage 4C is presented in the next section; downstream scoring remains deferred.


In [ ]:
specialists = integrate_specialist_evidence(REPORT_DIR / "stage2", REPORT_DIR / "stage4", DATA_ROOT)
specialists.write_reports(REPORT_DIR / "stage4" / "4b")

for measure, count in specialists.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(pd.Series(specialists.manifest["litchemplast_methods"], name="compounds").rename_axis("LitChemPlast match method").to_frame())
display(specialists.compounds.groupby("fcchumon_evidence_status").size().rename("compounds").to_frame())
display(specialists.anomalies[[
    "vega_row_no", "identity_reason", "litchemplast_match_method",
    "litchemplast_linked_record_count", "litchemplast_unique_publication_count",
    "fcchumon_evidence_status", "prediction_exclusion_reason"
]])
print("Stage 4B complete. Independent source evidence is retained; identity-review predictions remain unavailable.")


## Stage 4C — Analytical-platform and regulatory-list annotations

NORMAN SusDat predictions are matched by the canonical full InChIKey to the MS-ready key. Candidate selection follows nonmissing analytical-field count, maximum platform probability and source order for ties. These are **predictions of analytical amenability**, not observed detections; the historical LC/GC probability summaries are display metadata and do not enter scoring.

GIC association follows full-key lookup when available, CID bridges from PubChemLite and SusDat, then the resolved chemical name. This frozen inventory has no queryable InChIKeys. Name fallback is flagged for review; multiple CIDs and SIDs remain explicit. The pinned library unions tags through a global identifier index; shared names can propagate tags from other SIDs. Added tags and all contributing source rows are retained, and `gic_shared_identifier_tag_expansion` marks affected associations for review. An association does not establish identity or current legal applicability. Unqueried compounds are distinct from queried compounds with no match.

The inventory and both bridge sources have frozen input hashes. **The metadata CSV's historical byte hash and the historical GIC package version were not recorded.** Current metadata bytes and the GIC repository revision are pinned. Used effective metadata fields are checked against the frozen GIC audit by the separate regression command. Original values, source rows, attempted paths and selection/exclusion reasons are exported under `results/stage4/4c/`.

All 15 upstream identity reviews remain unresolved. Their 57 QSAR and 12 Pov/LRTP fields stay unavailable, with original reasons and source links retained. Independent NORMAN and GIC annotations use the Stage 2 identity. No downstream scoring is performed.


In [ ]:
annotations = integrate_annotations(REPORT_DIR / "stage4" / "4b", DATA_ROOT)
annotations.write_reports(REPORT_DIR / "stage4" / "4c")

for measure, count in annotations.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(annotations.compounds.groupby("norman_prediction_available").size().rename("compounds").to_frame())
display(annotations.compounds.groupby(
    ["gic_match_method", "gic_name_review_required", "gic_multiple_cids", "gic_multiple_sids", "gic_shared_identifier_tag_expansion"]
).size().rename("compounds").reset_index())
display(annotations.anomalies[[
    "inchikey", "vega_row_no", "qsar_identity_reason", "qsar_evidence_available", "lrtp_available",
    "norman_prediction_available", "gic_match_method", "gic_name_review_required", "gic_shared_identifier_tag_expansion"
]])
print("Metadata historical byte hash: not recorded; frozen effective-field validation is required.")
print("Stage 4C complete. Identity reviews remain pending; no downstream scoring performed.")


## Stage 5A — Persistence and mobility

Persistence uses the maximum usable IRFMN compartment assessment, with ADI ≥ 0.5. Mobility uses the maximum usable OPERA aquatic and OECD LRTP pathway score. A usable independent score, including zero, takes precedence over curated fallback. Missing evidence remains unavailable; it is not assigned zero. Exact field mappings, units, cutoffs and tie rules are declared in `plastprior/hazard_policy.json`.

**Two result roles.** Historical reproduction restores the frozen Stage 3 model inputs solely for comparison. The analysis table continues to withhold QSAR and Pov/LRTP evidence for the 15 identity-review compounds. Independent curated P/M evidence remains eligible, but none is available for this cohort. Exclusion reasons and the 58 original source links are retained.

GIC name and shared-identifier review flags remain visible as annotations and do not enter P/M scoring. NORMAN platform predictions also remain outside scoring. The GIC associations still require review; preserving historical values does not resolve these issues.

The summary distinguishes historical availability from analysis availability. Field-level changes, full evidence JSON, source-selection decisions and separate historical/analysis audits are exported under `results/stage5`. Historical agreement is checked separately with `plastprior.validation --stage hazard-pm`; executing this notebook alone does not establish equivalence. No B/T, exposure or aggregate priority scores are calculated.


In [ ]:
hazard = integrate_pm_hazard(REPORT_DIR / "stage4" / "4c")
hazard.write_reports(REPORT_DIR / "stage5")

display(hazard.summary)
review_keys = hazard.compounds.loc[
    hazard.compounds["qsar_identity_status"].eq("review_required"), "inchikey"
]
historical_review = hazard.historical.loc[
    hazard.historical["inchikey"].isin(review_keys),
    ["inchikey", "Persistence_score", "Mobility_score"]
].rename(columns={"Persistence_score": "historical_P", "Mobility_score": "historical_M"})
display(historical_review.merge(hazard.compounds[[
    "inchikey", "Persistence_score", "Mobility_score",
    "persistence_unavailable_reason", "mobility_unavailable_reason"
]], on="inchikey", validate="one_to_one"))
display(pd.Series({
    "GIC name matches requiring review": hazard.manifest["counts"]["gic_name_review_required"],
    "GIC shared-identifier tag expansions requiring review": hazard.manifest["counts"]["gic_tag_expansion_review_required"],
    "Identity-review source links retained": hazard.manifest["counts"]["identity_review_source_links"]
}, name="count").to_frame())
print("Stage 5A complete. GIC and NORMAN annotations do not enter P/M scoring.")


## Stage 5B — Bioaccumulation and toxicity

B selects the first usable BCF model in Arnot–Gobas, CAESAR, Meylan order (ADI ≥ 0.5), then takes the maximum of that score and the historical Cl/Br structural rule. T takes the maximum usable endpoint score. Ames has no ADI field in the frozen configuration; it still requires an eligible identity association. Usable independent evidence, including zero, precedes curated fallback. Missing evidence is not assigned zero. These are inherited screening rules, not independently validated regulatory classifications.

**Structure provenance.** The structural pathway uses `qsar_smiles`, generated locally from the Stage 2 input structure. It never uses the model-report `Basic | Smile_std`. All mappings and the ten text changes introduced by local standardisation are exported, with source representative IDs and RDKit version.

**Identity isolation.** The 15 identity-review compounds retain their independent local structures and curated observations, while all QSAR and Pov/LRTP predictions remain withheld from analysis. One has a usable structural B score and five have curated T fallback. Three T scores increase when a historical QSAR zero is replaced by curated evidence; the table below explicitly shows these source changes. Historical reproduction remains separate and is not eligible for downstream analysis.

GIC review flags remain unresolved annotations and do not enter B/T scoring. NORMAN platform predictions and toxicological labels also do not determine scores; the latter are audit context only. Full evidence, source-selection/exclusion reasons, 90 field-level identity impacts and separate historical/analysis audits are saved under `results/stage5/5b`. Run `python -m plastprior.validation --stage hazard-bt` with the documented arguments for historical verification. No aggregate hazard, exposure or priority score is calculated.


In [ ]:
hazard_bt = integrate_bt_hazard(REPORT_DIR / "stage5")
hazard_bt.write_reports(REPORT_DIR / "stage5" / "5b")

display(hazard_bt.summary)
review_changes = hazard_bt.impacts.loc[
    hazard_bt.impacts["field"].isin(["Bioaccumulation_score", "Toxicity_score"]),
    ["inchikey", "field", "historical_value", "analysis_value", "historical_basis", "analysis_basis", "change_reason"]
]
display(review_changes)
display(hazard_bt.structures.loc[
    hazard_bt.structures["text_changed_by_local_screening"],
    ["inchikey", "source_smiles_standard", "scoring_smiles", "structural_score"]
])
print("GIC review flags retained:", hazard_bt.manifest["counts"]["gic_name_review_required"],
      "name matches and", hazard_bt.manifest["counts"]["gic_tag_expansion_review_required"], "shared-identifier expansion.")
print("Stage 5B complete. P, B, M and T remain separate dimensions; no aggregate score is calculated.")


## Stage 6A — Exposure evidence and eligibility

Import fingerprinted historical production/market tables and verify their recorded raw-file hashes. These are frozen derived inputs; source transformations are not rerun. Every observation retains its native fields, source identifiers and exclusion reasons. Qualitative market evidence requires an accepted identity mapping before aggregation, independently of numeric availability.

GIC name-only associations remain under review. Inventory eligibility requires direct local tag support on at least one matched SID; metadata classifications retain the historical acquisition-hash limitation. Patent and PubMed queries retain unsuccessful and out-of-scope records. Query hits and inventory listings are proxies, not measured exposure.

The 15 unverified model associations remain isolated; independently supported exposure evidence is audited separately. No exposure scores or numeric maxima are calculated.


In [ ]:
exposure = integrate_exposure_evidence(REPORT_DIR / "stage5" / "5b", DATA_ROOT)
exposure.write_reports(REPORT_DIR / "stage6" / "6a")

display(exposure.summary)
display(pd.read_csv(REPORT_DIR / "stage6" / "6a" / "market_source_summary.csv"))
print("Exposure evidence audited; no exposure scores calculated.")


## Stage 6B — Exposure dimension assessment

Score five evidence dimensions: production/market scale, patent activity, PubMed query frequency, food-contact/human-detection evidence, and industrial-inventory association. Food contact and human detection remain separate binary contributions within one dimension. Historical reproduction retains the frozen selection and zero-filling rules for regression only. The analysis uses Stage 6A eligibility: ambiguous market identities, unresolved GIC name associations, invalid quantities and failed external queries cannot supply the affected evidence.

Missing evidence remains unavailable. FCChumon records with no valid query CAS or a rejected ambiguous-CAS association do not establish a zero detection indicator. A formal total is reported only when all five dimensions are available; the observed-component sum and coverage are retained separately, without renormalisation. A zero denotes the defined evidence indicator, not proof of no real-world exposure. The 15 model-review associations remain isolated, while independent eligible exposure evidence is retained. These results are screening proxies; no combined hazard-exposure priority or ranking is calculated.


In [ ]:
exposure_scores = integrate_exposure_scores(REPORT_DIR / "stage6" / "6a", DATA_ROOT)
exposure_scores.write_reports(REPORT_DIR / "stage6" / "6b")

display(exposure_scores.summary)
print(exposure_scores.manifest["counts"])
print("Review missing dimensions and evidence changes before interpreting exposure totals.")
